# Notebook 5 — Tablas Finales FSM + HMM + Clustering

**Inputs:** todos los outputs de NB1–NB4  
**Outputs:** `outputs/tabla1_*.csv` … `outputs/tabla4_*.csv`

In [ ]:
import os, numpy as np, random, pandas as pd
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
BASE_PATH = '.'
INPUTS  = os.path.join(BASE_PATH, 'data')
OUTPUTS = os.path.join(BASE_PATH, 'outputs')
ID_COL = 'email'; STATE_COL = 'state'; TIME_COL = 'timestamp'


In [ ]:
traj_full      = pd.read_csv(f"{OUTPUTS}/trajectories_full.csv")
clusters       = pd.read_csv(f"{OUTPUTS}/clusters_k4.csv")
z_long         = pd.read_csv(f"{OUTPUTS}/hmm_k4_hidden_states_long.csv")
z_summary_prop = pd.read_csv(f"{OUTPUTS}/hmm_k4_hidden_states_summary.csv")
Z_COLS = [c for c in z_summary_prop.columns if c.startswith('Z')]
print('Z_COLS:', Z_COLS)


## Tabla 1 — Composición FSM por estado latente Z

**Corrección aplicada:** merge por `email + t` para alinear correctamente el estado latente con el evento observable.

In [ ]:
# Merge correcto: email + índice temporal t
traj_sorted = traj_full.sort_values([ID_COL, TIME_COL]).copy()
traj_sorted['t'] = traj_sorted.groupby(ID_COL).cumcount()
fsm_z = traj_sorted.merge(z_long, on=[ID_COL, 't'])  # correcto: email + t

fsm_z_prop = fsm_z.groupby(['Z', STATE_COL]).size().unstack(fill_value=0)
fsm_z_prop = fsm_z_prop.div(fsm_z_prop.sum(axis=1), axis=0)
fsm_z_prop.round(3).to_csv(f"{OUTPUTS}/tabla1_fsm_por_z.csv")
print('Tabla 1 — Composición FSM por estado latente Z:')
fsm_z_prop.round(3)


## Tabla 2 — Estados latentes Z por alumno

In [ ]:
grades = pd.read_csv(f"{INPUTS}/grades_export_anon.csv")
grades.columns = [c.strip().lower() for c in grades.columns]
GRADE_COL = next((c for c in ['nota_final','nota','final_grade'] if c in grades.columns), None)

tabla2 = (z_summary_prop
    .merge(grades[['email', GRADE_COL]], on='email')
    .merge(clusters[['email','cluster']], on='email')
)
tabla2[['email'] + Z_COLS + [GRADE_COL, 'cluster']].round(3).to_csv(f"{OUTPUTS}/tabla2_z_por_alumno.csv", index=False)
print('Tabla 2:')
tabla2[['email'] + Z_COLS + [GRADE_COL, 'cluster']].round(3)


## Tabla 3 — Z por cluster | Tabla 4 — Síntesis modos cognitivos

In [ ]:
# Tabla 3
z_cluster = (z_summary_prop.merge(clusters[['email','cluster']], on='email')
             .groupby('cluster')[Z_COLS].mean())
z_cluster.round(3).to_csv(f"{OUTPUTS}/tabla3_z_por_cluster.csv")
print('Tabla 3 — Z por cluster:')
print(z_cluster.round(3))

# Tabla 4
tabla4 = pd.DataFrame({
    'Modo cognitivo': ['Lector / explorador de recursos','Navegador superficial',
                       'Aprendiz activo regulado','Pasivo / difuso'],
    'FSM dominante': ['REC','NAV','PRACT','OTHER'],
    'Fase temporal': ['Temprana','Temprana-media','Media-tardía','Tardía'],
    'Relación con rendimiento': ['Baja','Nula','Alta (significativa)','Baja']
}, index=['Z0','Z1','Z2','Z3'])
tabla4.to_csv(f"{OUTPUTS}/tabla4_sintesis_modos.csv")
print('\nTabla 4 — Síntesis modos cognitivos:')
print(tabla4)
